---
title: Tearsheet report
execute:
  enabled: true
---

A full quantstats-style strategy tearsheet, built entirely on qrt's own
stats/plot stack: `q.plot.report`. It lays out a chart column (cumulative
returns, EOY returns, return distributions, rolling risk diagnostics,
drawdown analysis, a monthly heatmap) alongside a table column (key metrics,
EOY returns, worst drawdowns) in a single scrollable Plotly figure. Every
panel is also available as a standalone chart, useful for embedding a single
diagnostic in a custom layout. These code cells are **executed live** by
Quarto every time the docs are built.

## Sample data

We reuse qrt's bundled sample datasets — AAPL as the "strategy" and SPY as the
benchmark — loaded offline via `q.data.datasets.load`, no network dependency —
sliced to the last decade (2016+):

In [1]:
import pandas as pd

import qrt as q

aapl = q.data.datasets.load("aapl")
spy = q.data.datasets.load("spy")

returns = pd.concat(
    {
        "AAPL": aapl["close"].pct_change(),
        "SPY": spy["close"].pct_change(),
    },
    axis=1,
).dropna().loc["2016":]
strategy = returns["AAPL"]
benchmark = returns["SPY"]

returns.tail()

,AAPL,SPY
datetime,,
2026-07-14,-0.007721,0.003551
2026-07-15,0.040145,0.003964
2026-07-16,0.017588,-0.005419
2026-07-17,0.001440,-0.009897
2026-07-20,-0.021424,-0.001614


## Full tearsheet: `q.plot.report`

Passing a `benchmark` overlays it throughout (cumulative-returns variants,
EOY bars/table, monthly distribution, rolling volatility, and adds a rolling
beta panel), and adds a *vs. Benchmark* section (Beta, Alpha, Correlation,
R², Information Ratio, ...) to the key metrics table. The metrics table
covers the full quantstats set — grouped into Returns, Risk-Adjusted, Risk,
Drawdown, Trading, Period Returns, Best / Worst, and Win Rates sections —
or a condensed subset with `mode="basic"`:


In [2]:
fig = q.plot.report(strategy, benchmark=benchmark, title="AAPL vs. SPY Tearsheet")
fig.show()

## Without a benchmark

Drop `benchmark` for a strategy-only tearsheet — the benchmark-relative rows
(volatility-matched returns, rolling beta) are omitted automatically:

In [3]:
fig = q.plot.report(strategy, title="AAPL Tearsheet")
fig.show()

## Individual panels

Every panel in `q.plot.report` is also a standalone function, taking the same
`returns`/`benchmark`/`return_type`/`periods_per_year`/`rf` arguments:

- `q.plot.metrics_table` — key performance metrics (Sharpe, Sortino, Calmar,
  Max Drawdown, ...), plus benchmark-relative rows when `benchmark` is given.
- `q.plot.cumulative_returns` — growth-of-$1 curve, with a `scale` of
  `"linear"`, `"log"`, or `"volatility_matched"`.
- `q.plot.eoy_returns` — end-of-year returns bar chart.
- `q.plot.monthly_distribution` — histogram of monthly returns.
- `q.plot.daily_returns` — daily return bars.
- `q.plot.rolling_volatility`, `q.plot.rolling_sharpe`, `q.plot.rolling_sortino`
  — rolling risk/reward diagnostics over a moving window.
- `q.plot.rolling_beta` — rolling beta to a benchmark over two window lengths.
- `q.plot.worst_drawdowns` — equity curve with the worst drawdown episodes shaded.
- `q.plot.return_quantiles` — box plots of daily/weekly/monthly/quarterly/yearly returns.

For example, the key metrics table on its own:

In [4]:
fig = q.plot.metrics_table(strategy, benchmark=benchmark)
fig.show()

...and the worst-drawdowns panel:

In [5]:
fig = q.plot.worst_drawdowns(strategy, top=5)
fig.show()